In [3]:
%pip install langchain langchain-community langchain-openai langchain-huggingface python-dotenv
!pip install -U langchain-core langchain langchain-community langchain-openai langchain-huggingface python-dotenv
print("done")

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


done



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
import sys
import langchain_core

print("Python:")
print(sys.executable)

print("\nLangChain Core:")
print(langchain_core.__version__)

print("\nLoaded from:")
print(langchain_core.__file__)

Python:
d:\text to sql\.venv\Scripts\python.exe

LangChain Core:
1.6.6

Loaded from:
d:\text to sql\.venv\Lib\site-packages\langchain_core\__init__.py


In [6]:
import pkgutil
import langchain_core

print([
    m.name
    for m in pkgutil.iter_modules(langchain_core.__path__)
])

['_api', '_import_utils', '_security', 'agents', 'caches', 'callbacks', 'chat_history', 'chat_loaders', 'chat_sessions', 'cross_encoders', 'document_loaders', 'documents', 'embeddings', 'env', 'example_selectors', 'exceptions', 'globals', 'indexing', 'language_models', 'load', 'messages', 'output_parsers', 'outputs', 'prompt_values', 'prompts', 'rate_limiters', 'retrievers', 'runnables', 'stores', 'structured_query', 'sys_info', 'tools', 'tracers', 'utils', 'vectorstores', 'version']


In [7]:
import sys

!{sys.executable} -m pip show langchain-core

'd:\text' is not recognized as an internal or external command,
operable program or batch file.


In [8]:
!{sys.executable} -m pip install --force-reinstall langchain-core

'd:\text' is not recognized as an internal or external command,
operable program or batch file.


In [4]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.utilities import SQLDatabase
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv

from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint, HuggingFacePipeline

C:\Users\Jafar tamboli\AppData\Local\Temp\ipykernel_24932\1751766401.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.utilities import SQLDatabase


In [5]:
from langchain_core.output_parsers import StrOutputParser

print("Working!")

Working!


In [6]:
db = SQLDatabase.from_uri("sqlite:///Chinook.db", sample_rows_in_table_info=0)

def get_schema(_):
    return db.get_table_info()

def run_query(query):
    print(f'Running query: {query} \n\n')
    return db.run(query)

In [7]:
print(get_schema(''))


CREATE TABLE "Album" (
	"AlbumId" INTEGER NOT NULL, 
	"Title" NVARCHAR(160) NOT NULL, 
	"ArtistId" INTEGER NOT NULL, 
	PRIMARY KEY ("AlbumId"), 
	FOREIGN KEY("ArtistId") REFERENCES "Artist" ("ArtistId")
)


CREATE TABLE "Artist" (
	"ArtistId" INTEGER NOT NULL, 
	"Name" NVARCHAR(120), 
	PRIMARY KEY ("ArtistId")
)


CREATE TABLE "Customer" (
	"CustomerId" INTEGER NOT NULL, 
	"FirstName" NVARCHAR(40) NOT NULL, 
	"LastName" NVARCHAR(20) NOT NULL, 
	"Company" NVARCHAR(80), 
	"Address" NVARCHAR(70), 
	"City" NVARCHAR(40), 
	"State" NVARCHAR(40), 
	"Country" NVARCHAR(40), 
	"PostalCode" NVARCHAR(10), 
	"Phone" NVARCHAR(24), 
	"Fax" NVARCHAR(24), 
	"Email" NVARCHAR(60) NOT NULL, 
	"SupportRepId" INTEGER, 
	PRIMARY KEY ("CustomerId"), 
	FOREIGN KEY("SupportRepId") REFERENCES "Employee" ("EmployeeId")
)


CREATE TABLE "Employee" (
	"EmployeeId" INTEGER NOT NULL, 
	"LastName" NVARCHAR(20) NOT NULL, 
	"FirstName" NVARCHAR(20) NOT NULL, 
	"Title" NVARCHAR(30), 
	"ReportsTo" INTEGER, 
	"BirthDate" 

LLM connection

In [8]:
def get_llm(load_from_hugging_face=False):
    if load_from_hugging_face:
        llm = ChatHuggingFace(
            model_name="google/flan-t5-xxl",
            task="text2text-generation",
            model_kwargs={"temperature": 0, "max_length": 512},
        )
        return ChatHuggingFace(llm-llm)

    return ChatOpenAI(model="gpt-3.5-turbo", temperature=0)


def write_sql_query(llm):
    template = """Based on the table schema below, write a SQL query that would answer the user's question:
    {schema}

    Question: {question}
    SQL Query:"""

    prompt = ChatPromptTemplate.from_messages(
        [

            ("system","Given an input question, convert it to a SQL query that can be run against a database. Only output the SQL query, do not include any explanations or any prefix or sufix text."),
        ]
    )

    return(
        RunnablePassthrough.assign(schema = get_schema)
        | prompt
        | llm
        | StrOutputParser()
    )

In [ ]:
def answer_user_query(query,llm):
    template = """Based on the schema below, question, sql query, and sql response, write a natural language response:
    {schema}

    Question: {question}
    SQL Query: {query}
    SQL Response: {response}"""

    prompt_response = ChatPromptTemplate.from_messages(
        [       
            ("system",
             "Given an input question and SQL response, convert it to a natural language answer, No pre-amble.",
            ),
            ("human", template),
        ]       
    )
